# Microsoft Fabric SQL Endpoint Metadata Refresh Playbook

This notebook is a Fabric-native workflow for diagnosing and forcing metadata synchronization between a Lakehouse and its SQL analytics endpoint.

## Why this notebook exists
The transcript you shared describes intermittent cases where Spark/Lakehouse writes are not immediately visible in the SQL endpoint. Microsoft Fabric now provides an official REST API for on-demand metadata refresh (`refreshMetadata`), so this notebook prioritizes supported APIs and operational guardrails.

## What this notebook does
1. Detects workspace/lakehouse context from the active Fabric notebook session.
2. Resolves SQL endpoint ID from Lakehouse metadata.
3. Triggers on-demand metadata refresh via Fabric REST API.
4. Handles long-running operations (LRO) and collects final per-table sync status.
5. Surfaces failures (`Failure`) and no-op states (`NotRun`) for troubleshooting.


## Transcript findings mapped to official behavior

- SQL endpoint metadata sync is automatic, but not always instant.
- Under normal conditions, lag is typically less than one minute, but can be longer depending on workload and table layout.
- On-demand refresh is supported both from the Fabric UI and the official REST API.
- Unsupported Delta-to-SQL mappings can cause missing columns or partial table representation in SQL endpoint.
- `NotRun` is often expected: it usually means a table did not require a refresh in that run.


In [ ]:
import json
import re
import time
from datetime import datetime, timezone

import pandas as pd
import requests

try:
    from notebookutils import notebookutils
except Exception:
    import notebookutils

BASE_URL = "https://api.fabric.microsoft.com"

# Optional manual overrides (keep as None for automatic context detection)
WORKSPACE_ID_OVERRIDE = None
LAKEHOUSE_ID_OVERRIDE = None

# Runtime knobs
RECREATE_TABLES = False
REQUEST_TIMEOUT_MINUTES = 20
POLL_SECONDS = 5
MAX_POLLS = 240  # 20 minutes at 5-second polling


In [ ]:
def _context_to_dict():
    ctx = notebookutils.runtime.context

    if isinstance(ctx, dict):
        return ctx

    for accessor in ("asDict", "toDict"):
        if hasattr(ctx, accessor):
            try:
                value = getattr(ctx, accessor)()
                if isinstance(value, dict):
                    return value
            except Exception:
                pass

    if hasattr(ctx, "__dict__") and isinstance(ctx.__dict__, dict) and ctx.__dict__:
        return dict(ctx.__dict__)

    txt = str(ctx).strip()
    if txt.startswith("{") and txt.endswith("}"):
        try:
            parsed = json.loads(txt)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

    keys = [
        "currentWorkspaceId",
        "currentWorkspaceName",
        "defaultLakehouseId",
        "defaultLakehouseName",
        "defaultLakehouseWorkspaceId",
        "defaultLakehouseWorkspaceName",
        "currentNotebookId",
        "currentNotebookName",
    ]
    out = {}
    for key in keys:
        if hasattr(ctx, key):
            out[key] = getattr(ctx, key)
    return out


context = _context_to_dict()

WORKSPACE_ID = WORKSPACE_ID_OVERRIDE or context.get("defaultLakehouseWorkspaceId") or context.get("currentWorkspaceId")
WORKSPACE_NAME = context.get("defaultLakehouseWorkspaceName") or context.get("currentWorkspaceName")
LAKEHOUSE_ID = LAKEHOUSE_ID_OVERRIDE or context.get("defaultLakehouseId")
LAKEHOUSE_NAME = context.get("defaultLakehouseName")

print("Workspace:", WORKSPACE_NAME, WORKSPACE_ID)
print("Lakehouse:", LAKEHOUSE_NAME, LAKEHOUSE_ID)

if not WORKSPACE_ID or not LAKEHOUSE_ID:
    raise ValueError(
        "Could not auto-detect workspace/lakehouse IDs. "
        "Attach a default Lakehouse or set WORKSPACE_ID_OVERRIDE and LAKEHOUSE_ID_OVERRIDE."
    )


In [ ]:
def _utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def _extract_error_message(response):
    try:
        payload = response.json()
    except Exception:
        return response.text

    if isinstance(payload, dict):
        if isinstance(payload.get("error"), dict):
            return payload["error"].get("message") or json.dumps(payload["error"])
        if "message" in payload:
            return payload["message"]
        return json.dumps(payload)

    return str(payload)


def _get_bearer_token():
    token = notebookutils.credentials.getToken("pbi")
    if not token:
        raise RuntimeError("Failed to acquire token via notebookutils.credentials.getToken('pbi').")
    return token


def fabric_request(method, path, *, token=None, params=None, body=None, expected_statuses=None):
    if token is None:
        token = _get_bearer_token()

    response = requests.request(
        method=method.upper(),
        url=f"{BASE_URL}{path}",
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        },
        params=params,
        json=body,
        timeout=120,
    )

    if expected_statuses and response.status_code not in expected_statuses:
        raise RuntimeError(
            f"{method.upper()} {path} failed with HTTP {response.status_code}: "
            f"{_extract_error_message(response)}"
        )

    return response


def response_json_or_none(response):
    try:
        return response.json()
    except Exception:
        return None


In [ ]:
def get_lakehouse_details(workspace_id, lakehouse_id, token=None):
    path = f"/v1/workspaces/{workspace_id}/lakehouses/{lakehouse_id}"
    response = fabric_request("GET", path, token=token, expected_statuses={200})
    return response_json_or_none(response)


def resolve_sql_endpoint_context(workspace_id, lakehouse_id, token=None):
    details = get_lakehouse_details(workspace_id, lakehouse_id, token=token)
    if not isinstance(details, dict):
        raise RuntimeError("Lakehouse details response is empty or not JSON.")

    props = details.get("properties", {})
    sql_props = props.get("sqlEndpointProperties", {})
    sql_endpoint_id = sql_props.get("id")

    if not sql_endpoint_id:
        raise RuntimeError("Could not resolve sqlEndpointProperties.id from Lakehouse metadata.")

    return {
        "workspaceId": workspace_id,
        "lakehouseId": lakehouse_id,
        "lakehouseName": details.get("displayName"),
        "sqlEndpointId": sql_endpoint_id,
        "sqlEndpointProvisioningStatus": sql_props.get("provisioningStatus"),
        "sqlEndpointConnectionString": sql_props.get("connectionString"),
    }


token = _get_bearer_token()
endpoint_context = resolve_sql_endpoint_context(WORKSPACE_ID, LAKEHOUSE_ID, token=token)
endpoint_context


In [ ]:
def _operation_id_from_headers(headers):
    operation_id = headers.get("x-ms-operation-id")
    if operation_id:
        return operation_id

    location = headers.get("Location") or headers.get("location")
    if location:
        match = re.search(r"/operations/([0-9a-fA-F-]+)", location)
        if match:
            return match.group(1)

    return None


def get_operation_state(operation_id, token=None):
    path = f"/v1/operations/{operation_id}"
    response = fabric_request("GET", path, token=token, expected_statuses={200})
    return response_json_or_none(response), response.headers


def get_operation_result(operation_id, token=None):
    path = f"/v1/operations/{operation_id}/result"
    response = fabric_request("GET", path, token=token, expected_statuses={200})
    return response_json_or_none(response)


def wait_for_operation(operation_id, *, poll_seconds=5, max_polls=240, token=None):
    for _ in range(max_polls):
        state, headers = get_operation_state(operation_id, token=token)
        status = (state or {}).get("status")

        if status == "Succeeded":
            return get_operation_result(operation_id, token=token), state
        if status == "Failed":
            raise RuntimeError(f"Operation {operation_id} failed: {json.dumps(state, indent=2)}")

        retry_after = headers.get("Retry-After") or headers.get("retry-after")
        wait_seconds = int(retry_after) if retry_after and str(retry_after).isdigit() else poll_seconds
        time.sleep(wait_seconds)

    raise TimeoutError(f"Operation {operation_id} did not finish after {max_polls} polls.")


def refresh_sql_endpoint_metadata(
    workspace_id,
    sql_endpoint_id,
    *,
    recreate_tables=False,
    timeout_minutes=20,
    poll_seconds=5,
    max_polls=240,
    token=None,
):
    path = f"/v1/workspaces/{workspace_id}/sqlEndpoints/{sql_endpoint_id}/refreshMetadata"
    body = {
        "recreateTables": bool(recreate_tables),
        "timeout": {
            "timeUnit": "Minutes",
            "value": int(timeout_minutes),
        },
    }

    response = fabric_request("POST", path, token=token, body=body, expected_statuses={200, 202})
    payload = response_json_or_none(response)

    if response.status_code == 200:
        return {
            "startedAtUtc": _utc_now_iso(),
            "mode": "Immediate",
            "operationId": None,
            "operationState": {"status": "Succeeded"},
            "result": payload,
        }

    operation_id = _operation_id_from_headers(response.headers)
    if not operation_id:
        raise RuntimeError("refreshMetadata returned 202 but no operation ID was found in headers.")

    result, operation_state = wait_for_operation(
        operation_id,
        poll_seconds=poll_seconds,
        max_polls=max_polls,
        token=token,
    )

    return {
        "startedAtUtc": _utc_now_iso(),
        "mode": "LRO",
        "operationId": operation_id,
        "operationState": operation_state,
        "result": result,
    }


refresh_run = refresh_sql_endpoint_metadata(
    workspace_id=endpoint_context["workspaceId"],
    sql_endpoint_id=endpoint_context["sqlEndpointId"],
    recreate_tables=RECREATE_TABLES,
    timeout_minutes=REQUEST_TIMEOUT_MINUTES,
    poll_seconds=POLL_SECONDS,
    max_polls=MAX_POLLS,
    token=token,
)

refresh_run["mode"], refresh_run["operationId"], refresh_run["operationState"]


In [ ]:
def table_sync_dataframe(refresh_result):
    if not isinstance(refresh_result, dict):
        return pd.DataFrame()

    rows = refresh_result.get("value", [])
    if not isinstance(rows, list):
        return pd.DataFrame()

    df = pd.json_normalize(rows, sep="_")
    if df.empty:
        return df

    for col in ["startDateTime", "endDateTime", "lastSuccessfulSyncDateTime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

    if "startDateTime" in df.columns and "endDateTime" in df.columns:
        df["durationSeconds"] = (df["endDateTime"] - df["startDateTime"]).dt.total_seconds()

    ordered_cols = [
        "tableName",
        "status",
        "lastSuccessfulSyncDateTime",
        "startDateTime",
        "endDateTime",
        "durationSeconds",
        "error_errorCode",
        "error_message",
    ]
    existing = [c for c in ordered_cols if c in df.columns]
    others = [c for c in df.columns if c not in existing]
    return df[existing + others].sort_values(["status", "tableName"], na_position="last")


sync_df = table_sync_dataframe(refresh_run["result"])
sync_df


In [ ]:
if sync_df.empty:
    print("No per-table rows were returned.")
else:
    print(sync_df["status"].value_counts(dropna=False).to_string())
    failures = sync_df[sync_df["status"] == "Failure"]

    if failures.empty:
        print("\nNo failed tables in this run.")
    else:
        print("\nFailed tables:")
        display(failures[[c for c in ["tableName", "status", "error_errorCode", "error_message"] if c in failures.columns]])

    not_run = sync_df[sync_df["status"] == "NotRun"]
    if not not_run.empty:
        print(f"\n{len(not_run)} table(s) returned NotRun (usually already in sync).")


## How to interpret results

- `Success`: Table metadata was synchronized in this run.
- `NotRun`: Refresh logic skipped table (commonly already in sync).
- `Failure`: Sync failed for that table. Inspect `error_errorCode` and `error_message`.

### Common root causes
- Delta table columns with unsupported SQL endpoint mappings (for example, complex/unsupported types).
- Table is not in the Lakehouse `/Tables` path.
- High small-file churn or heavy partition/file fragmentation delaying metadata discovery.
- SQL endpoint provisioning is not in `Success` state.

### Escalation pattern
1. Validate table location and schema compatibility.
2. Trigger on-demand refresh (this notebook/API).
3. Optimize table maintenance strategy (`OPTIMIZE`, file/partition hygiene).
4. If issue persists, capture payload/errors and open Microsoft support case.


## Optional Spark probe block

Use this to create a small probe table in Spark and then rerun the refresh cell to verify end-to-end metadata propagation.

```python
probe_table = "sql_sync_probe"
spark.sql(f"DROP TABLE IF EXISTS {probe_table}")
spark.sql(f"CREATE TABLE {probe_table} AS SELECT current_timestamp() AS created_utc")
spark.sql(f"SELECT * FROM {probe_table}").show()
```


## Official references (used for this notebook)

- SQL analytics endpoint overview (Lakehouse): https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-sql-analytics-endpoint
- SQL analytics endpoint performance considerations: https://learn.microsoft.com/en-us/fabric/data-warehouse/sql-analytics-endpoint-performance
- Refresh SQL endpoint metadata API: https://learn.microsoft.com/en-us/rest/api/fabric/sqlendpoint/items/refresh-sql-endpoint-metadata
- Get Lakehouse API (includes `sqlEndpointProperties.id`): https://learn.microsoft.com/en-us/rest/api/fabric/lakehouse/items/get-lakehouse
- LRO state API: https://learn.microsoft.com/en-us/rest/api/fabric/core/long-running-operations/get-operation-state
- LRO result API: https://learn.microsoft.com/en-us/rest/api/fabric/core/long-running-operations/get-operation-result
- Data type mapping and unsupported behavior: https://learn.microsoft.com/en-us/fabric/data-warehouse/data-types
- SQL endpoint limitations and metadata discovery constraints: https://learn.microsoft.com/en-us/fabric/data-warehouse/limitations
- Notebook utilities (`runtime.context`, `credentials.getToken`): https://learn.microsoft.com/en-us/fabric/data-engineering/notebook-utilities
- SemPy `FabricRestClient` reference (context only): https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric?view=semantic-link-python
